# State Store

> SQLite-backed workflow state storage for persistence across restarts.

In [ ]:
#| default_exp state_store

In [ ]:
#| export
import json
import sqlite3
import uuid
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Any, Optional, List
from contextlib import contextmanager

## SessionSummary

Lightweight metadata for a workflow session, suitable for listing sessions in a management UI without loading full state blobs.

In [ ]:
#| export
@dataclass
class SessionSummary:
    """Lightweight metadata for a single workflow session."""
    flow_id:str # Workflow identifier
    session_id:str # Session identifier
    label:Optional[str] # Human-readable label, or None to defer to a generator
    current_step:Optional[str] # Current StepFlow step ID, or None
    created_at:str # ISO timestamp string from SQLite CURRENT_TIMESTAMP
    updated_at:str # ISO timestamp string from SQLite CURRENT_TIMESTAMP
    state_size_bytes:int # Length of state_json column — cheap size hint

## SQLiteWorkflowStateStore

SQLite-backed implementation of workflow state storage. Accepts a plain `session_id: str`
for all operations — no web framework dependency.

Key features:

- State persists across application restarts
- Supports step-specific state storage for progress restoration
- Uses JSON serialization for complex state objects
- Composite key pattern: `flow_id:session_id` for multi-workflow support

In [ ]:
#| export
class SQLiteWorkflowStateStore:
    """SQLite-backed workflow state storage for persistence across restarts."""
    
    def __init__(
        self,
        db_path:Path # Path to SQLite database file
    ):
        """Initialize the state store and create or migrate the schema."""
        self.db_path = Path(db_path)
        self.db_path.parent.mkdir(parents=True, exist_ok=True)
        self._init_db()
    
    @contextmanager
    def _get_connection(self): # Context manager yielding a SQLite connection
        """Get a database connection with commit-on-success and guaranteed close."""
        conn = sqlite3.connect(self.db_path)
        conn.row_factory = sqlite3.Row
        try:
            yield conn
            conn.commit()
        finally:
            conn.close()
    
    def _init_db(
        self,
    ) -> None:
        """Initialize the schema and run idempotent migrations."""
        with self._get_connection() as conn:
            conn.execute("""
                CREATE TABLE IF NOT EXISTS workflow_state (
                    flow_session_key TEXT PRIMARY KEY,
                    current_step TEXT,
                    state_json TEXT DEFAULT '{}',
                    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
                    updated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
                )
            """)
            conn.execute("""
                CREATE INDEX IF NOT EXISTS idx_workflow_state_updated
                ON workflow_state(updated_at)
            """)
            # Idempotent migration: add label column if missing.
            cursor = conn.execute("PRAGMA table_info(workflow_state)")
            columns = {row[1] for row in cursor.fetchall()}
            if "label" not in columns:
                conn.execute("ALTER TABLE workflow_state ADD COLUMN label TEXT")
    
    def _make_key(
        self,
        flow_id:str, # Workflow identifier
        session_id:str # Session identifier string
    ) -> str: # Composite "flow_id:session_id" key for storage
        """Build the composite primary key from flow and session IDs."""
        return f"{flow_id}:{session_id}"
    
    def get_current_step(
        self,
        flow_id:str, # Workflow identifier
        session_id:str # Session identifier string
    ) -> Optional[str]: # Current step ID, or None if no row exists
        """Get the current step ID for a session."""
        key = self._make_key(flow_id, session_id)
        with self._get_connection() as conn:
            row = conn.execute(
                "SELECT current_step FROM workflow_state WHERE flow_session_key = ?",
                (key,)
            ).fetchone()
            return row["current_step"] if row else None
    
    def set_current_step(
        self,
        flow_id:str, # Workflow identifier
        session_id:str, # Session identifier string
        step_id:str # Step ID to set as current
    ) -> None:
        """Set the current step ID for a session, upserting the row if needed."""
        key = self._make_key(flow_id, session_id)
        with self._get_connection() as conn:
            conn.execute("""
                INSERT INTO workflow_state (flow_session_key, current_step)
                VALUES (?, ?)
                ON CONFLICT(flow_session_key) DO UPDATE SET
                    current_step = excluded.current_step,
                    updated_at = CURRENT_TIMESTAMP
            """, (key, step_id))
    
    def get_state(
        self,
        flow_id:str, # Workflow identifier
        session_id:str # Session identifier string
    ) -> Dict[str, Any]: # Full session state dictionary
        """Get the full state dictionary for a session."""
        key = self._make_key(flow_id, session_id)
        with self._get_connection() as conn:
            row = conn.execute(
                "SELECT state_json FROM workflow_state WHERE flow_session_key = ?",
                (key,)
            ).fetchone()
            if row and row["state_json"]:
                return json.loads(row["state_json"])
            return {}
    
    def update_state(
        self,
        flow_id:str, # Workflow identifier
        session_id:str, # Session identifier string
        updates:Dict[str, Any] # Top-level state keys to merge in
    ) -> None:
        """Merge updates into the session state, upserting the row if needed."""
        key = self._make_key(flow_id, session_id)
        current_state = self.get_state(flow_id, session_id)
        current_state.update(updates)
        state_json = json.dumps(current_state)
        with self._get_connection() as conn:
            conn.execute("""
                INSERT INTO workflow_state (flow_session_key, state_json)
                VALUES (?, ?)
                ON CONFLICT(flow_session_key) DO UPDATE SET
                    state_json = excluded.state_json,
                    updated_at = CURRENT_TIMESTAMP
            """, (key, state_json))
    
    def clear_state(
        self,
        flow_id:str, # Workflow identifier
        session_id:str # Session identifier string
    ) -> None:
        """Delete the session row entirely. Idempotent."""
        key = self._make_key(flow_id, session_id)
        with self._get_connection() as conn:
            conn.execute(
                "DELETE FROM workflow_state WHERE flow_session_key = ?",
                (key,)
            )
    
    def get_step_state(
        self,
        flow_id:str, # Workflow identifier
        session_id:str, # Session identifier string
        step_id:str # Step identifier
    ) -> Dict[str, Any]: # State dictionary for that step
        """Get the state dictionary for a specific step."""
        state = self.get_state(flow_id, session_id)
        step_key = f"_step_{step_id}"
        return state.get(step_key, {})
    
    def update_step_state(
        self,
        flow_id:str, # Workflow identifier
        session_id:str, # Session identifier string
        step_id:str, # Step identifier
        updates:Dict[str, Any] # Step state keys to merge in
    ) -> None:
        """Merge updates into a specific step's state."""
        step_key = f"_step_{step_id}"
        current_step_state = self.get_step_state(flow_id, session_id, step_id)
        current_step_state.update(updates)
        self.update_state(flow_id, session_id, {step_key: current_step_state})
    
    def clear_step_state(
        self,
        flow_id:str, # Workflow identifier
        session_id:str, # Session identifier string
        step_id:str # Step identifier
    ) -> None:
        """Remove a specific step's state while preserving the rest of the session."""
        step_key = f"_step_{step_id}"
        state = self.get_state(flow_id, session_id)
        if step_key in state:
            del state[step_key]
            key = self._make_key(flow_id, session_id)
            state_json = json.dumps(state)
            with self._get_connection() as conn:
                conn.execute("""
                    UPDATE workflow_state SET state_json = ?, updated_at = CURRENT_TIMESTAMP
                    WHERE flow_session_key = ?
                """, (state_json, key))
    
    def create_session(
        self,
        flow_id:str, # Workflow identifier
        session_id:Optional[str]=None, # Pre-chosen session ID (auto-generated UUID4 if None)
        label:Optional[str]=None # Optional human-readable label
    ) -> str: # The created session_id
        """Insert a new empty session row and return its session_id."""
        if session_id is None:
            session_id = str(uuid.uuid4())
        key = self._make_key(flow_id, session_id)
        with self._get_connection() as conn:
            conn.execute(
                "INSERT INTO workflow_state (flow_session_key, label) VALUES (?, ?)",
                (key, label)
            )
        return session_id
    
    def session_exists(
        self,
        flow_id:str, # Workflow identifier
        session_id:str # Session identifier string
    ) -> bool: # True if a row exists for the given session
        """Check whether a session row exists."""
        key = self._make_key(flow_id, session_id)
        with self._get_connection() as conn:
            row = conn.execute(
                "SELECT 1 FROM workflow_state WHERE flow_session_key = ?",
                (key,)
            ).fetchone()
            return row is not None
    
    def get_session_summary(
        self,
        flow_id:str, # Workflow identifier
        session_id:str # Session identifier string
    ) -> Optional[SessionSummary]: # Session metadata, or None if not found
        """Return lightweight metadata for a single session."""
        key = self._make_key(flow_id, session_id)
        with self._get_connection() as conn:
            row = conn.execute("""
                SELECT current_step, label, created_at, updated_at, length(state_json) AS size
                FROM workflow_state WHERE flow_session_key = ?
            """, (key,)).fetchone()
        if row is None:
            return None
        return SessionSummary(
            flow_id=flow_id,
            session_id=session_id,
            label=row["label"],
            current_step=row["current_step"],
            created_at=row["created_at"],
            updated_at=row["updated_at"],
            state_size_bytes=row["size"] or 0,
        )
    
    def list_sessions(
        self,
        flow_id:str, # Workflow identifier to list sessions for
        order_by:str="updated_at", # Sort column: "updated_at", "created_at", or "label"
        descending:bool=True # Sort direction
    ) -> List[SessionSummary]: # All sessions for this flow, ordered
        """List all sessions for a flow as lightweight SessionSummary records."""
        if order_by not in ("updated_at", "created_at", "label"):
            raise ValueError(f"Invalid order_by: {order_by!r}")
        direction = "DESC" if descending else "ASC"
        prefix = f"{flow_id}:"
        with self._get_connection() as conn:
            rows = conn.execute(f"""
                SELECT flow_session_key, current_step, label, created_at, updated_at,
                       length(state_json) AS size
                FROM workflow_state
                WHERE flow_session_key LIKE ?
                ORDER BY {order_by} {direction}
            """, (f"{prefix}%",)).fetchall()
        return [
            SessionSummary(
                flow_id=flow_id,
                session_id=r["flow_session_key"][len(prefix):],
                label=r["label"],
                current_step=r["current_step"],
                created_at=r["created_at"],
                updated_at=r["updated_at"],
                state_size_bytes=r["size"] or 0,
            )
            for r in rows
        ]
    
    def set_session_label(
        self,
        flow_id:str, # Workflow identifier
        session_id:str, # Session identifier string
        label:Optional[str] # New label, or None to clear
    ) -> None:
        """Update the session label. No-op if the session does not exist."""
        key = self._make_key(flow_id, session_id)
        with self._get_connection() as conn:
            conn.execute("""
                UPDATE workflow_state
                SET label = ?, updated_at = CURRENT_TIMESTAMP
                WHERE flow_session_key = ?
            """, (label, key))
    
    def delete_session(
        self,
        flow_id:str, # Workflow identifier
        session_id:str # Session identifier string
    ) -> None:
        """Delete a session row. Alias of `clear_state` for symmetry with the session API."""
        self.clear_state(flow_id, session_id)

## Tests

In [ ]:
import tempfile, os

# Create a temporary database for testing
_tmp = tempfile.NamedTemporaryFile(suffix='.db', delete=False)
_tmp_path = _tmp.name
_tmp.close()
store = SQLiteWorkflowStateStore(_tmp_path)

In [ ]:
# Step tracking
assert store.get_current_step("flow1", "sess1") is None

store.set_current_step("flow1", "sess1", "selection")
assert store.get_current_step("flow1", "sess1") == "selection"

store.set_current_step("flow1", "sess1", "decomposition")
assert store.get_current_step("flow1", "sess1") == "decomposition"

# Different sessions are independent
assert store.get_current_step("flow1", "sess2") is None

print("Step tracking tests passed!")

Step tracking tests passed!


In [ ]:
# State CRUD
assert store.get_state("flow1", "sess1") == {}

store.update_state("flow1", "sess1", {"count": 5, "name": "test"})
state = store.get_state("flow1", "sess1")
assert state["count"] == 5
assert state["name"] == "test"

# Merge semantics
store.update_state("flow1", "sess1", {"count": 10})
state = store.get_state("flow1", "sess1")
assert state["count"] == 10
assert state["name"] == "test"  # preserved

# Clear
store.clear_state("flow1", "sess1")
assert store.get_state("flow1", "sess1") == {}

print("State CRUD tests passed!")

State CRUD tests passed!


In [ ]:
# Step-specific state
assert store.get_step_state("flow1", "sess1", "selection") == {}

store.update_step_state("flow1", "sess1", "selection", {"sources": ["a", "b"]})
step_state = store.get_step_state("flow1", "sess1", "selection")
assert step_state["sources"] == ["a", "b"]

# Different steps are independent
store.update_step_state("flow1", "sess1", "decomposition", {"segments": [1, 2, 3]})
assert store.get_step_state("flow1", "sess1", "selection")["sources"] == ["a", "b"]
assert store.get_step_state("flow1", "sess1", "decomposition")["segments"] == [1, 2, 3]

# Clear step state
store.clear_step_state("flow1", "sess1", "selection")
assert store.get_step_state("flow1", "sess1", "selection") == {}
assert store.get_step_state("flow1", "sess1", "decomposition")["segments"] == [1, 2, 3]  # preserved

print("Step-specific state tests passed!")

Step-specific state tests passed!


In [ ]:
# Flow and session isolation
store.clear_state("flow1", "sess1")
store.update_state("flow1", "sess1", {"flow": 1})
store.update_state("flow2", "sess1", {"flow": 2})
store.update_state("flow1", "sess2", {"flow": 1, "session": 2})

assert store.get_state("flow1", "sess1")["flow"] == 1
assert store.get_state("flow2", "sess1")["flow"] == 2
assert store.get_state("flow1", "sess2")["session"] == 2

print("Isolation tests passed!")

Isolation tests passed!


In [ ]:
# Migration: new install already has the label column.
# (Re-initializing should be idempotent — safe to call twice.)
store._init_db()
with store._get_connection() as _c:
    _cols = {r[1] for r in _c.execute("PRAGMA table_info(workflow_state)").fetchall()}
assert "label" in _cols, f"label column missing: {_cols}"

print("Migration (fresh install) tests passed!")

In [ ]:
# Migration: existing pre-label DB gets upgraded in place and preserves data.
_legacy = tempfile.NamedTemporaryFile(suffix='.db', delete=False)
_legacy_path = _legacy.name
_legacy.close()

# Hand-create the OLD schema (no label column) and insert a row.
_conn = sqlite3.connect(_legacy_path)
_conn.execute("""
    CREATE TABLE workflow_state (
        flow_session_key TEXT PRIMARY KEY,
        current_step TEXT,
        state_json TEXT DEFAULT '{}',
        created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
        updated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    )
""")
_conn.execute(
    "INSERT INTO workflow_state (flow_session_key, current_step, state_json) VALUES (?, ?, ?)",
    ("flowL:legacy1", "selection", '{"x": 1}')
)
_conn.commit()
_conn.close()

# Open with the new store — should migrate in place.
legacy_store = SQLiteWorkflowStateStore(_legacy_path)

# Legacy row still readable.
assert legacy_store.get_state("flowL", "legacy1") == {"x": 1}
assert legacy_store.get_current_step("flowL", "legacy1") == "selection"

# Label column was added and legacy row gets NULL label.
summary = legacy_store.get_session_summary("flowL", "legacy1")
assert summary is not None
assert summary.label is None
assert summary.current_step == "selection"
assert summary.state_size_bytes > 0

# list_sessions surfaces the legacy row.
sessions = legacy_store.list_sessions("flowL")
assert len(sessions) == 1
assert sessions[0].session_id == "legacy1"

os.unlink(_legacy_path)
print("Migration (existing DB) tests passed!")

In [ ]:
# create_session: auto-UUID, explicit ID, and with label.
store.clear_state("flowS", "s1")  # ensure clean slate

new_id = store.create_session("flowS")
assert isinstance(new_id, str) and len(new_id) == 36  # UUID4 string form
assert store.session_exists("flowS", new_id)

# Explicit ID.
store.create_session("flowS", session_id="s1", label="hand-named")
assert store.session_exists("flowS", "s1")

summary = store.get_session_summary("flowS", "s1")
assert summary is not None
assert summary.label == "hand-named"
assert summary.current_step is None  # not set yet
assert summary.state_size_bytes >= 2  # at least '{}' default

# session_exists is False for a missing session.
assert not store.session_exists("flowS", "nope")

# Duplicate create_session raises (INSERT conflict).
import sqlite3 as _sqlite3
try:
    store.create_session("flowS", session_id="s1")
    assert False, "expected IntegrityError on duplicate create_session"
except _sqlite3.IntegrityError:
    pass

print("create_session / session_exists tests passed!")

In [ ]:
# list_sessions: multiple sessions, ordering, and scoping by flow_id.
# Start from a clean flow to avoid interference with earlier tests.
for _sid in [s.session_id for s in store.list_sessions("flowLS")]:
    store.delete_session("flowLS", _sid)

# Create three sessions with controlled updated_at ordering.
import time
store.create_session("flowLS", session_id="alpha", label="Alpha")
time.sleep(0.01)
store.create_session("flowLS", session_id="beta", label="Beta")
time.sleep(0.01)
store.create_session("flowLS", session_id="gamma", label="Gamma")

# Default ordering: updated_at DESC → gamma, beta, alpha.
sessions = store.list_sessions("flowLS")
assert [s.session_id for s in sessions] == ["gamma", "beta", "alpha"], \
    f"unexpected order: {[s.session_id for s in sessions]}"

# Ascending order.
sessions_asc = store.list_sessions("flowLS", descending=False)
assert [s.session_id for s in sessions_asc] == ["alpha", "beta", "gamma"]

# Order by label.
sessions_by_label = store.list_sessions("flowLS", order_by="label", descending=False)
assert [s.label for s in sessions_by_label] == ["Alpha", "Beta", "Gamma"]

# Listing a different flow returns none.
assert store.list_sessions("flowLS_other") == []

# Invalid order_by raises.
try:
    store.list_sessions("flowLS", order_by="state_json")
    assert False, "expected ValueError on invalid order_by"
except ValueError:
    pass

print("list_sessions tests passed!")

In [ ]:
# set_session_label: update and clear.
store.create_session("flowR", session_id="r1", label="Initial")
assert store.get_session_summary("flowR", "r1").label == "Initial"

store.set_session_label("flowR", "r1", "Renamed")
assert store.get_session_summary("flowR", "r1").label == "Renamed"

# Clear the label.
store.set_session_label("flowR", "r1", None)
assert store.get_session_summary("flowR", "r1").label is None

# Setting a label on a nonexistent session is a no-op (doesn't raise, doesn't create row).
store.set_session_label("flowR", "does_not_exist", "Ghost")
assert not store.session_exists("flowR", "does_not_exist")

print("set_session_label tests passed!")

In [ ]:
# delete_session: removes the row and is idempotent.
store.create_session("flowD", session_id="d1", label="Doomed")
store.update_state("flowD", "d1", {"payload": [1, 2, 3]})
assert store.session_exists("flowD", "d1")
assert store.get_state("flowD", "d1")["payload"] == [1, 2, 3]

store.delete_session("flowD", "d1")
assert not store.session_exists("flowD", "d1")
assert store.get_state("flowD", "d1") == {}
assert store.get_session_summary("flowD", "d1") is None

# Idempotent: deleting a nonexistent session does not raise.
store.delete_session("flowD", "d1")
store.delete_session("flowD", "never_existed")

print("delete_session tests passed!")

In [ ]:
# get_session_summary: integrates current_step, state size, and existing state.
store.create_session("flowM", session_id="m1", label="Meta")
store.set_current_step("flowM", "m1", "decomposition")
store.update_state("flowM", "m1", {"segments": list(range(50))})

summary = store.get_session_summary("flowM", "m1")
assert summary is not None
assert summary.flow_id == "flowM"
assert summary.session_id == "m1"
assert summary.label == "Meta"
assert summary.current_step == "decomposition"
assert summary.state_size_bytes > 50  # serialized list takes some bytes
assert summary.created_at is not None
assert summary.updated_at is not None

# Missing session returns None.
assert store.get_session_summary("flowM", "missing") is None

print("get_session_summary tests passed!")

In [ ]:
# Cleanup
os.unlink(_tmp_path)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()